In [ ]:
from pathlib import Path
import pandas as pd

from src.loop import SelfImprovingLoop, LoopConfig, LoopAgents
from src.agent_profiles import (
    Agent,
    base_agent_options,
    proposer_options,
    skill_generator_options,
    prompt_generator_options,
)
from src.agent_profiles.skill_generator import get_project_root
from src.registry import ProgramManager
from src.schemas import (
    AgentResponse,
    ProposerResponse,
    ToolGeneratorResponse,
    PromptGeneratorResponse,
)
# import pickle as pkl

# with open("../results/opsu_full.pkl", "rb") as f:
#     graded_data = pkl.load(f)
# dataset_path = Path("~/mnt/shared-resources-sentient-research/salah_resources/datasets/officeqa/")
# data = pd.read_csv(dataset_path / "officeqa.csv")
# hard_data = data[data['difficulty'] == 'hard']

# train_data = [(row.question, row.answer) for _, row in hard_data.head(5).iterrows()]
# val_data = [(row.question, row.answer) for _, row in hard_data.tail(3).iterrows()]

In [ ]:
data = pd.read_csv('../.dataset/train_set.csv')

train = data.sample(20, random_state=42)
val = data.drop(train.index).sample(5, random_state=31)

train_data = [(row.question, row.ground_truth) for _, row in train.iterrows()]
val_data = [(row.question, row.ground_truth) for _, row in val.iterrows()]

In [ ]:
agents = LoopAgents(
    base=Agent(base_agent_options, AgentResponse),
    proposer=Agent(proposer_options, ProposerResponse),
    skill_generator=Agent(skill_generator_options, ToolGeneratorResponse),
    prompt_generator=Agent(prompt_generator_options, PromptGeneratorResponse),
)
manager = ProgramManager(cwd=get_project_root())

In [ ]:
config = LoopConfig(
    max_iterations=10,
    frontier_size=3,
    no_improvement_limit=5,
    concurrency=4,
    cache_enabled=True,
    reset_feedback=True,
)

In [ ]:
loop = SelfImprovingLoop(config, agents, manager, train_data, val_data)
result = await loop.run()

# Cell 5: Results
print(f"Best: {result.best_program} ({result.best_score:.2%})")
print(f"Frontier: {result.frontier}")